# Logistic Regression – Airline Customer Satisfaction Prediction

Fully executed notebook with preprocessing, encoding, logistic regression, evaluation metrics, coefficient interpretation, recommendations, and limitations.

In [1]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, classification_report

df = pd.read_csv('invistico_Airline.csv')
print("Shape:", df.shape)
df.head()


Shape: (129880, 22)


,satisfaction,Customer Type,Age,Type of Travel,Class,Flight Distance,Seat comfort,Departure/Arrival time convenient,Food and drink,Gate location,Inflight wifi service,Inflight entertainment,Online support,Ease of Online booking,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Online boarding,Departure Delay in Minutes,Arrival Delay in Minutes
0,satisfied,Loyal Customer,65,Personal Travel,Eco,265,0,0,0,2,2,4,2,3,3,0,3,5,3,2,0,0.0
1,satisfied,Loyal Customer,47,Personal Travel,Business,2464,0,0,0,3,0,2,2,3,4,4,4,2,3,2,310,305.0
2,satisfied,Loyal Customer,15,Personal Travel,Eco,2138,0,0,0,3,2,0,2,2,3,3,4,4,4,2,0,0.0
3,satisfied,Loyal Customer,60,Personal Travel,Eco,623,0,0,0,3,3,4,3,1,1,0,1,4,1,3,0,0.0
4,satisfied,Loyal Customer,70,Personal Travel,Eco,354,0,0,0,3,4,3,4,2,2,0,2,4,2,5,0,0.0


In [2]:

print(df['satisfaction'].value_counts())


satisfaction
satisfied       71087
dissatisfied    58793
Name: count, dtype: int64


In [3]:

X = df.drop('satisfaction', axis=1)
y = (df['satisfaction'].astype(str).str.lower()=='satisfied').astype(int)

cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)
pred = model.predict(X_test)

cm = confusion_matrix(y_test, pred)
print("Confusion Matrix\n", cm)
print("Accuracy:", accuracy_score(y_test,pred))
print("Precision:", precision_score(y_test,pred))
print("Recall:", recall_score(y_test,pred))
print(classification_report(y_test,pred))


Confusion Matrix
 [[ 9544  2215]
 [ 2233 11984]]
Accuracy: 0.8287650138589467
Precision: 0.8440030988097753
Recall: 0.8429345150172329
              precision    recall  f1-score   support

           0       0.81      0.81      0.81     11759
           1       0.84      0.84      0.84     14217

    accuracy                           0.83     25976
   macro avg       0.83      0.83      0.83     25976
weighted avg       0.83      0.83      0.83     25976



In [4]:

feature_names = model.named_steps['preprocessor'].get_feature_names_out()
coef = model.named_steps['classifier'].coef_[0]

coef_df = pd.DataFrame({'Feature':feature_names,'Coefficient':coef})
coef_df['Abs'] = coef_df['Coefficient'].abs()

print("Top Positive Drivers")
print(coef_df.sort_values('Coefficient', ascending=False)[['Feature','Coefficient']].head(10))

print("\nTop Negative Drivers")
print(coef_df.sort_values('Coefficient', ascending=True)[['Feature','Coefficient']].head(10))


Top Positive Drivers
                                Feature  Coefficient
7           num__Inflight entertainment     0.972683
18    cat__Customer Type_Loyal Customer     0.826756
22                  cat__Class_Business     0.423151
2                     num__Seat comfort     0.392571
10                num__On-board service     0.388175
13                 num__Checkin service     0.354962
9           num__Ease of Online booking     0.333694
11                num__Leg room service     0.308907
20  cat__Type of Travel_Business travel     0.267379
15                 num__Online boarding     0.181688

Top Negative Drivers
                                   Feature  Coefficient
19    cat__Customer Type_disloyal Customer    -1.046129
21     cat__Type of Travel_Personal Travel    -0.486751
24                     cat__Class_Eco Plus    -0.349142
3   num__Departure/Arrival time convenient    -0.327559
4                      num__Food and drink    -0.296607
23                          cat__Class


## Business Recommendations

- Improve inflight entertainment and seat comfort.
- Strengthen loyalty programs.
- Improve digital booking and check-in experience.
- Focus on service quality and economy-class experience.

## Limitations

- Logistic Regression assumes linearity in log-odds.
- Additional models such as Random Forest and XGBoost should be evaluated.
